# 05B - Sequence Memory PyTorch Lab

**Guiding question:** can one PyTorch sequence pipeline handle both continuous forecasting and padded next-token learning without hiding shape, causality, or gradient contracts?

> **What you finished last time:** 05A measured why long memory needs a better route.
> **What this notebook delivers:** a forecasting LSTM and a padding-safe token LSTM, both trained from a fresh kernel.
> **Prerequisite for the next notebook:** the helper contract in `sequence_helpers.py`, reused by the piano capstone.

| Part | Failure first | PyTorch contract |
|---|---|---|
| 1 | random splits leak future structure | sliding windows plus temporal split |
| 2 | a tensor alone does not define batching | `Dataset` and `DataLoader` |
| 3 | padded final states describe padding | packed sequences with real lengths |
| 4 | pad tokens can dominate loss | `ignore_index` plus explicit reduction |
| 5 | sequence code can accidentally see the future | causal-invariance test |
| 6 | recurrent gradients can spike | clip after backward, before step |


In [ ]:
# -- Setup and canonical helpers --------------------------------------------------
import math
import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import (
    pack_padded_sequence,
    pad_packed_sequence,
    pad_sequence,
)
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from sequence_helpers import backward_step, seed_everything, trainable_parameter_count

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams.update({"figure.facecolor": "#1a1a2e", "axes.facecolor": "#1a1a2e"})
print(f"Device: {device}")


## 1 - Forecasting makes the sequence contract visible

```mermaid
flowchart LR
    A["Low-noise signal"] --> B["Sliding windows"]
    B --> C["Temporal split"]
    C --> D["Dataset and DataLoader"]
    D --> E["nn.LSTM"]
    E --> F["Next-value prediction"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

**Predict:** why split by time instead of shuffling every window before the split? Adjacent windows overlap; a random split can place almost-identical histories on both sides.


In [ ]:
# -- Low-noise series, sliding windows, and temporal split ------------------------
points = 720
time = torch.linspace(0, 8 * math.pi, points)
series = torch.sin(time) + 0.25 * torch.sin(3 * time)
window = 24

windows = torch.stack([series[i : i + window] for i in range(points - window)])
targets = torch.stack([series[i + window] for i in range(points - window)])
split = int(0.75 * len(windows))
X_train, X_valid = windows[:split], windows[split:]
y_train, y_valid = targets[:split], targets[split:]

print(f"Train windows: {tuple(X_train.shape)}")
print(f"Valid windows: {tuple(X_valid.shape)}")
print(f"Boundary indices: train ends at {split - 1}, validation starts at {split}")
print("Takeaway: validation occurs strictly later in time.")


In [ ]:
# -- Dataset and DataLoader: make one feature explicit ----------------------------
class WindowDataset(Dataset):
    def __init__(self, inputs, labels):
        self.inputs = inputs.unsqueeze(-1).float()
        self.labels = labels.unsqueeze(-1).float()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        return self.inputs[index], self.labels[index]

train_loader = DataLoader(WindowDataset(X_train, y_train), batch_size=64, shuffle=True)
valid_loader = DataLoader(WindowDataset(X_valid, y_valid), batch_size=128, shuffle=False)
batch_x, batch_y = next(iter(train_loader))
print(f"Batch input: {tuple(batch_x.shape)} = (batch, time, feature)")
print(f"Batch target: {tuple(batch_y.shape)} = (batch, 1)")


In [ ]:
# -- First nn.LSTM model: predict from the final sequence output ------------------
class ForecastLSTM(nn.Module):
    def __init__(self, hidden_size=24):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, values):
        outputs, _ = self.lstm(values)
        return self.head(outputs[:, -1])

forecast_model = ForecastLSTM().to(device)
optimizer = torch.optim.Adam(forecast_model.parameters(), lr=0.02)
loss_fn = nn.MSELoss()
train_history = []

for epoch in range(45):
    forecast_model.train()
    epoch_losses = []
    for inputs, labels in train_loader:
        predictions = forecast_model(inputs.to(device))
        loss = loss_fn(predictions, labels.to(device))
        backward_step(loss, forecast_model, optimizer)
        epoch_losses.append(float(loss.detach()))
    train_history.append(np.mean(epoch_losses))

forecast_model.eval()
valid_predictions = []
valid_targets = []
with torch.inference_mode():
    for inputs, labels in valid_loader:
        valid_predictions.append(forecast_model(inputs.to(device)).cpu())
        valid_targets.append(labels)
valid_predictions = torch.cat(valid_predictions).squeeze(1)
valid_targets = torch.cat(valid_targets).squeeze(1)
valid_mse = torch.mean((valid_predictions - valid_targets) ** 2).item()
baseline_mse = torch.mean((X_valid[:, -1] - y_valid) ** 2).item()

print(f"Parameters: {trainable_parameter_count(forecast_model):,}")
print(f"Persistence baseline MSE: {baseline_mse:.6f}")
print(f"LSTM validation MSE:      {valid_mse:.6f}")
print("Takeaway: the complete window improves on simply repeating the last value.")
assert np.isfinite(valid_mse)


In [ ]:
# -- Forecast evidence: loss and later-in-time predictions ------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_history, color="#60a5fa")
axes[0].set(title="Forecast training loss", xlabel="Epoch", ylabel="MSE")
axes[1].plot(valid_targets[:120], label="actual", color="#34d399")
axes[1].plot(valid_predictions[:120], label="predicted", color="#f97316")
axes[1].set(title="Later validation segment", xlabel="Step")
axes[1].legend()
plt.show()


### That works for equal windows. Language batches are not equal.

Forecast inputs had one real value at every position. Token sequences need integer addresses, shifted targets, padding, and real lengths.


## 2 - Token IDs and shifted targets

```mermaid
flowchart LR
    A["token IDs x0...xT-1"] --> B["nn.Embedding"]
    B --> C["packed LSTM"]
    C --> D["vocabulary logits"]
    D --> E["targets x1...xT"]
    E --> F["padding-aware loss"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Token IDs, shifted targets, and padding -------------------------------------
PAD = "<pad>"
motifs = [
    "D3 A3 D4 F4 A4",
    "D3 A3 D4 F4",
    "D3 A3 C4",
    "D3 F3 A3 D4 F4 A4",
]
vocabulary = [PAD] + sorted({token for motif in motifs for token in motif.split()})
token_to_id = {token: index for index, token in enumerate(vocabulary)}
pad_id = token_to_id[PAD]

encoded = [torch.tensor([token_to_id[token] for token in motif.split()]) for motif in motifs]
inputs = [row[:-1] for row in encoded]
targets = [row[1:] for row in encoded]
lengths = torch.tensor([len(row) for row in inputs])
padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=pad_id)
padded_targets = pad_sequence(targets, batch_first=True, padding_value=pad_id)
mask = padded_inputs.ne(pad_id)

print(f"Vocabulary: {vocabulary}")
print(f"Inputs shape:  {tuple(padded_inputs.shape)}")
print(f"Targets shape: {tuple(padded_targets.shape)}")
print(f"Lengths: {lengths.tolist()}")
print("First input/target pair:")
print(" ", padded_inputs[0].tolist(), "->", padded_targets[0].tolist())


In [ ]:
# -- Embedding lookup: IDs are addresses, not numeric magnitudes -----------------
embedding = nn.Embedding(len(vocabulary), embedding_dim=6, padding_idx=pad_id)
embedded = embedding(padded_inputs)
selected_row = embedding.weight[padded_inputs[0, 1]]
exact_lookup = torch.equal(embedded[0, 1], selected_row)

print(f"(B, T) IDs -> (B, T, D) vectors: {tuple(embedded.shape)}")
print(f"Lookup equals selected table row: {exact_lookup}")
print("Takeaway: token ID 5 is an address; it is not five times token ID 1.")
assert exact_lookup


## 3 - Pack before recurrence

Without packing, the final output for a short row belongs to `<pad>`. Packing lets the LSTM stop each row at its last real token.

**Predict:** after unpacking, should padded output positions contain learned recurrent values or zeros?


In [ ]:
# -- Packed sequences: final state means final real token -------------------------
inspection_lstm = nn.LSTM(input_size=6, hidden_size=8, batch_first=True)
packed = pack_padded_sequence(
    embedded,
    lengths.cpu(),
    batch_first=True,
    enforce_sorted=False,
)
packed_outputs, (h_n, c_n) = inspection_lstm(packed)
unpacked, unpacked_lengths = pad_packed_sequence(
    packed_outputs,
    batch_first=True,
    total_length=padded_inputs.size(1),
)

padded_values = unpacked.masked_select(~mask.unsqueeze(-1))
print(f"Unpacked outputs: {tuple(unpacked.shape)}")
print(f"Final hidden:     {tuple(h_n.shape)} = (layers, batch, hidden)")
print(f"Final cell:       {tuple(c_n.shape)}")
print(f"Returned lengths: {unpacked_lengths.tolist()}")
print(f"Maximum absolute padded output: {padded_values.abs().max().item():.6f}")
assert torch.allclose(padded_values, torch.zeros_like(padded_values))


In [ ]:
# -- Verify final hidden equals each row's last real output -----------------------
row_indices = torch.arange(len(lengths))
last_real_outputs = unpacked[row_indices, lengths - 1]
final_state_match = torch.allclose(last_real_outputs, h_n[-1], atol=1e-6)
print(f"Final hidden matches each last real timestep: {final_state_match}")
print("Takeaway: packing protects sequence meaning, not just compute.")
assert final_state_match


## 4 - Vocabulary logits and padding-aware loss

```mermaid
flowchart LR
    A["B x T token IDs"] --> B["B x T x D embeddings"]
    B --> C["B x T x H recurrent outputs"]
    C --> D["B x T x V raw logits"]
    D --> E["flatten real targets"]
    E --> F["mean over real tokens"]
    style A fill:#1e3a8a,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style F fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```


In [ ]:
# -- Tiny token model: embedding -> packed LSTM -> vocabulary logits -------------
class MotifLSTM(nn.Module):
    def __init__(self, vocab_size, pad_id, embedding_dim=12, hidden_size=20):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids, sequence_lengths):
        vectors = self.embedding(token_ids)
        packed = pack_padded_sequence(
            vectors,
            sequence_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        packed_outputs, state = self.lstm(packed)
        outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=token_ids.size(1),
        )
        return self.head(outputs), state

token_model = MotifLSTM(len(vocabulary), pad_id).to(device)
logits, state = token_model(padded_inputs.to(device), lengths)
print(f"Logits: {tuple(logits.shape)} = (batch, time, vocabulary)")
print(f"Hidden state: {tuple(state[0].shape)}")


In [ ]:
# -- Wrong versus correct loss: padding is bookkeeping, not a target -------------
flat_logits = logits.reshape(-1, len(vocabulary))
flat_targets = padded_targets.to(device).reshape(-1)
per_position = nn.functional.cross_entropy(
    flat_logits,
    flat_targets,
    reduction="none",
)
wrong_loss = per_position.mean()

real_target_mask = flat_targets.ne(pad_id)
explicit_masked_loss = per_position[real_target_mask].sum() / real_target_mask.sum()
ignore_index_loss = nn.functional.cross_entropy(
    flat_logits,
    flat_targets,
    ignore_index=pad_id,
)

print(f"Wrong all-position mean:       {wrong_loss.item():.6f}")
print(f"Explicit real-token reduction: {explicit_masked_loss.item():.6f}")
print(f"ignore_index reduction:        {ignore_index_loss.item():.6f}")
print(f"Correct reductions match: {torch.allclose(explicit_masked_loss, ignore_index_loss)}")
assert torch.allclose(explicit_masked_loss, ignore_index_loss)


## 5 - Prove causality instead of assuming it

A unidirectional LSTM should not let a changed future token alter logits at earlier positions. We will change only the last real input and compare the prefix.


In [ ]:
# -- Causal-invariance proof: future edits cannot change earlier logits ----------
causal_model = MotifLSTM(len(vocabulary), pad_id).to(device).eval()
reference = padded_inputs[:1].clone()
changed_future = reference.clone()
last_index = lengths[0].item() - 1
changed_future[0, last_index] = token_to_id["C4"]

with torch.inference_mode():
    reference_logits, _ = causal_model(reference.to(device), lengths[:1])
    changed_logits, _ = causal_model(changed_future.to(device), lengths[:1])

prefix_difference = (
    reference_logits[:, :last_index] - changed_logits[:, :last_index]
).abs().max().item()
edited_position_difference = (
    reference_logits[:, last_index] - changed_logits[:, last_index]
).abs().max().item()

print(f"Maximum earlier-logit difference: {prefix_difference:.8f}")
print(f"Edited-position difference:       {edited_position_difference:.8f}")
print("Takeaway: future token changes leave earlier unidirectional logits invariant.")
assert prefix_difference < 1e-7
assert edited_position_difference > 0


## 6 - Train with clipping in the correct place

```mermaid
flowchart LR
    A["forward"] --> B["padding-aware loss"]
    B --> C["backward"]
    C --> D["measure and clip gradients"]
    D --> E["optimizer step"]
    style A fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style B fill:#b45309,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style C fill:#1d4ed8,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style D fill:#b91c1c,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
    style E fill:#15803d,stroke:#e2e8f0,stroke-width:2px,color:#ffffff
```

Clipping before `backward()` clips nothing. Clipping after `step()` is too late.


In [ ]:
# -- Full token training loop with visible pre- and post-clip norms --------------
seed_everything(42)
token_model = MotifLSTM(len(vocabulary), pad_id).to(device)
optimizer = torch.optim.Adam(token_model.parameters(), lr=0.03)
loss_history = []
norm_history = []
clip_norm = 1.0

for epoch in range(120):
    token_model.train()
    logits, _ = token_model(padded_inputs.to(device), lengths)
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, len(vocabulary)),
        padded_targets.to(device).reshape(-1),
        ignore_index=pad_id,
    )
    pre_clip, post_clip = backward_step(loss, token_model, optimizer, clip_norm)
    loss_history.append(float(loss.detach()))
    norm_history.append((pre_clip, post_clip))

clipped_steps = sum(pre > clip_norm for pre, _ in norm_history)
print(f"Initial loss: {loss_history[0]:.6f}")
print(f"Final loss:   {loss_history[-1]:.6f}")
print(f"Steps above clip threshold: {clipped_steps}/{len(norm_history)}")
print(f"Final recorded norms: pre={norm_history[-1][0]:.4f}, post={norm_history[-1][1]:.4f}")
print("Takeaway: clipping is observable stabilization, not a hidden ritual.")
assert loss_history[-1] < loss_history[0]


### Your turn - tighten the clip threshold

Change only `clip_norm_to_test`. Predict whether more or fewer steps will be clipped. Do not change the learning rate in the same experiment.


In [ ]:
# -- Your turn: inspect one known training control --------------------------------
# CHANGE THIS: try 0.25 or 2.0
clip_norm_to_test = 0.5
would_clip = sum(pre > clip_norm_to_test for pre, _ in norm_history)
print(f"At threshold {clip_norm_to_test:.2f}, {would_clip}/{len(norm_history)} recorded steps would clip.")
print("Check: lowering the threshold cannot reduce the number of clipped recorded steps.")
assert would_clip >= clipped_steps


## 7 - Scorecard and handoff

| Contract | Evidence |
|---|---|
| temporal validation | validation windows occur strictly after training windows |
| `(B, T, F)` | forecast batches expose all three axes |
| shifted targets | each input token predicts the next token |
| packed sequences | final hidden equals each last real output |
| padding-aware loss | explicit reduction equals `ignore_index` |
| causal recurrence | future edit leaves earlier logits unchanged |
| clipping order | pre- and post-clip norms are recorded |

**Built and measured:** forecasting, token IDs, embeddings, packing, vocabulary logits, masked loss, causal invariance, clipping.

**Explained or illustrated:** why temporal splits matter and why padding is bookkeeping.

**Named and out of scope:** bidirectional recurrence, attention, and production checkpointing.

Next, [05C](05C-cinematic-piano-memory.ipynb) reuses the canonical helper contract and asks you to hear the memory difference.
